# Day 11: Rigorous 3D-CNN Evaluation & Error Analysis 🛰️

## 1. Objective and Engineering Context
On **Day 10**, we completed supervised training of the `Hyperspectral3DCNN` model on the Salinas dataset, saving the best checkpoint based on validation loss to `outputs/checkpoints/best_3d_cnn.pt`.

Today (**Day 11**), we conduct a **rigorous, scientific evaluation** on the held-out **Test Set** (8,120 samples, representing 15% of all active labeled pixels).

### Why a Strict Held-Out Test Set Matters:
1. **Preventing Optimistic Bias**: During training, the validation set was used to select the best checkpoint. If we evaluated final performance on the validation set, our reported metrics would suffer from subtle checkpoint-selection bias.
2. **True Generalization Assessment**: The test set was **strictly quarantined**—never touched for gradient updates, optimizer steps, learning rate schedules, or checkpoint selection.
3. **Multiclass Land-Cover Task**: The Salinas benchmark assesses agricultural crop classification across 16 land-cover classes (Broccoli, Celery, Corn, Grapes, Lettuce, Vineyard, etc.). It is an agricultural land-cover benchmark, not crop-disease diagnosis.

## 2. Checkpoint Verification and Dataset Loading
We verify that the saved checkpoint from Day 10 exists, load its architecture configuration, and inspect its metadata.

In [1]:
import sys
from pathlib import Path

# Resolve project root dynamically whether run from root or notebooks/
current_dir = Path.cwd().resolve()
if (current_dir / "data" / "raw" / "Salinas_corrected.mat").is_file():
    PROJECT_ROOT = current_dir
elif (current_dir.parent / "data" / "raw" / "Salinas_corrected.mat").is_file():
    PROJECT_ROOT = current_dir.parent
else:
    PROJECT_ROOT = Path("..").resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import scipy.io as sio
import torch

from terraspectra.data import (
    SalinasLoader,
    SALINAS_CLASS_NAMES,
    create_dataloaders,
)
from terraspectra.models import (
    Hyperspectral3DCNN,
    Hyperspectral3DCNNConfig,
    load_checkpoint,
    select_device,
    evaluate_model,
    plot_confusion_matrix,
    EvaluationReport,
)

print(f"Project Root:    {PROJECT_ROOT}")
device = select_device("auto")
print(f"Selected Device: {device}")

# Verify checkpoint file exists
checkpoint_file = PROJECT_ROOT / "outputs" / "checkpoints" / "best_3d_cnn.pt"
assert checkpoint_file.is_file(), f"Checkpoint missing at {checkpoint_file}"
print(f"Verified Checkpoint: {checkpoint_file} ({checkpoint_file.stat().st_size / (1024*1024):.2f} MB)")

Project Root:    C:\Users\abhis\Desktop\Tetraspectra
Selected Device: cpu
Verified Checkpoint: C:\Users\abhis\Desktop\Tetraspectra\outputs\checkpoints\best_3d_cnn.pt (0.29 MB)


## 3. Reconstructing Model & Loading Weights
We inspect the checkpoint payload, reconstruct the matching `Hyperspectral3DCNN` architecture, and load the trained weights into `model.eval()` mode.

In [2]:
# Inspect checkpoint metadata
payload = torch.load(checkpoint_file, map_location=device, weights_only=False)
model_config_dict = payload["model_config"]
print("Loaded Checkpoint Parameters:")
print(f"  Training Epoch:   {payload.get('epoch')}")
print(f"  Best Val Loss:    {payload.get('val_loss'):.4f}")
print(f"  Best Val Acc:     {payload.get('val_acc')*100:.2f}%")
print(f"  Spectral Depth:   {model_config_dict['spectral_depth']}")
print(f"  Spatial Window:   {model_config_dict['spatial_size']}x{model_config_dict['spatial_size']}")
print(f"  Number of Classes:{model_config_dict['num_classes']}")

# Reconstruct exact matching model
model_config = Hyperspectral3DCNNConfig(**model_config_dict)
model = Hyperspectral3DCNN(model_config)
load_checkpoint(checkpoint_file, model=model, device=device)
model.eval()
print("\n[SUCCESS] Model successfully restored and placed in eval mode.")

Loaded Checkpoint Parameters:
  Training Epoch:   1
  Best Val Loss:    0.0903
  Best Val Acc:     97.11%
  Spectral Depth:   15
  Spatial Window:   7x7
  Number of Classes:16

[SUCCESS] Model successfully restored and placed in eval mode.


## 4. Preparing the Held-Out Test DataLoader
Using `create_dataloaders` with the identical seed (`seed=42`), we reconstruct the exact 70/15/15% split.
- **Test Set**: Exactly 8,120 samples (15% of the 54,129 labeled pixels).
- **Label Mapping**: Contiguous 0-indexed targets $[0..15]$ representing original classes $[1..16]$.

In [3]:
# Load PCA-reduced cube or raw cube matching model spectral depth
if model_config.spectral_depth == 15:
    pca_data_path = PROJECT_ROOT / "data" / "processed" / "salinas_pca_reduced.npz"
    pca_npz = np.load(pca_data_path)
    cube = pca_npz["reduced_cube"]
else:
    raw_loader = SalinasLoader(raw_data_dir=PROJECT_ROOT / "data" / "raw")
    cube = raw_loader.load().cube

gt = sio.loadmat(PROJECT_ROOT / "data" / "raw" / "Salinas_gt.mat")["salinas_gt"]

# Recreate deterministic dataloaders
_, _, test_loader, meta = create_dataloaders(
    cube_or_dataset=cube,
    gt=gt,
    patch_size=model_config.spatial_size,
    batch_size=128,
    train_ratio=0.70,
    val_ratio=0.15,
    test_ratio=0.15,
    seed=42,
    stratify=True,
    exclude_unlabeled=True,
)

print(f"Test Samples to Evaluate: {meta.test_samples:,}")
print(f"Test Batches:             {len(test_loader)}")
print(f"Class Mapping (1..16 -> 0..15): {meta.num_classes} classes")

Test Samples to Evaluate: 8,120
Test Batches:             64
Class Mapping (1..16 -> 0..15): 16 classes


## 5. Executing Inference and Metric Computation
We run `evaluate_model` on the test loader using `torch.inference_mode()`.
- Calculates **Overall Accuracy (OA)**: $\text{OA} = \frac{\sum \text{True Positives}}{\text{Total Samples}}$
- Calculates **Average Accuracy (AA)**: $\text{AA} = \frac{1}{C} \sum_{c=1}^C \text{Recall}_c$
- Calculates **Macro F1-Score**: $\text{Macro } F_1 = \frac{1}{C} \sum_{c=1}^C F_{1, c}$
- Calculates **Cohen's Kappa ($\kappa$)**: Inter-rater agreement adjusting for chance agreement:
  $$\kappa = \frac{p_o - p_e}{1 - p_e}$$

In [4]:
# Evaluate model on held-out test data
eval_report = evaluate_model(
    model=model,
    test_loader=test_loader,
    device=device,
    label_to_idx=meta.label_to_idx,
    idx_to_label=meta.idx_to_label,
    class_names=[SALINAS_CLASS_NAMES[i] for i in range(1, 17)],
    checkpoint_path=str(checkpoint_file),
    checkpoint_epoch=payload.get("epoch", 1),
)

print("=" * 60)
print("       TERRASPECTRA 3D-CNN TEST EVALUATION BENCHMARK")
print("=" * 60)
print(f"Total Evaluated Test Samples: {eval_report.total_test_samples:,}")
print(f"Overall Accuracy (OA):        {eval_report.overall_accuracy * 100:.2f}%")
print(f"Average Accuracy (AA):        {eval_report.average_accuracy * 100:.2f}%")
print(f"Macro Precision:              {eval_report.macro_precision * 100:.2f}%")
print(f"Macro Recall:                 {eval_report.macro_recall * 100:.2f}%")
print(f"Macro F1-Score:               {eval_report.macro_f1 * 100:.2f}%")
print(f"Weighted F1-Score:            {eval_report.weighted_f1 * 100:.2f}%")
print(f"Cohen's Kappa (kappa):        {eval_report.cohen_kappa:.4f}")
print("=" * 60)

       TERRASPECTRA 3D-CNN TEST EVALUATION BENCHMARK
Total Evaluated Test Samples: 8,120
Overall Accuracy (OA):        97.02%
Average Accuracy (AA):        98.38%
Macro Precision:              98.70%
Macro Recall:                 98.38%
Macro F1-Score:               98.49%
Weighted F1-Score:            96.97%
Cohen's Kappa (kappa):        0.9668


## 6. Per-Class Performance Breakdown
A single overall accuracy score can conceal weaknesses on rare or spectrally confusing classes.
Below is the full per-class classification breakdown showing precision, recall, F1-score, and support.

In [5]:
print(f"{'Class Name':<30} {'ID':<4} {'Idx':<4} {'Prec (%)':<10} {'Recall (%)':<12} {'F1 (%)':<10} {'Support':<8}")
print("-" * 82)
for cm in eval_report.per_class_metrics:
    print(
        f"{cm.class_name:<30} "
        f"{cm.class_id:<4} "
        f"{cm.mapped_index:<4} "
        f"{cm.precision*100:<10.2f} "
        f"{cm.recall*100:<12.2f} "
        f"{cm.f1_score*100:<10.2f} "
        f"{cm.support:<8}"
    )
print("-" * 82)

Class Name                     ID   Idx  Prec (%)   Recall (%)   F1 (%)     Support 
----------------------------------------------------------------------------------
Broccoli_green_weeds_1         1    0    100.00     100.00       100.00     302     
Broccoli_green_weeds_2         2    1    100.00     100.00       100.00     559     
Fallow                         3    2    100.00     100.00       100.00     297     
Fallow_rough_plow              4    3    100.00     98.09        99.03      209     
Fallow_smooth                  5    4    99.01      100.00       99.50      401     
Stubble                        6    5    100.00     100.00       100.00     594     
Celery                         7    6    100.00     100.00       100.00     537     
Grapes_untrained               8    7    89.65      97.87        93.58      1690    
Soil_vinyard_develop           9    8    100.00     100.00       100.00     931     
Corn_senesced_green_weeds      10   9    99.59      99.39        99

## 7. Confusion Matrix Visualization
The confusion matrix plots ground-truth rows against predicted columns. Perfect predictions lie strictly on the diagonal.

In [6]:
# Save confusion matrix figure to outputs/figures/
fig_path = PROJECT_ROOT / "outputs" / "figures" / "salinas_confusion_matrix.png"

fig = plot_confusion_matrix(
    cm=eval_report.confusion_matrix,
    class_names=[cm.class_name for cm in eval_report.per_class_metrics],
    normalize=False,
    title=f"Salinas 3D-CNN Confusion Matrix (Test Set N={eval_report.total_test_samples:,}, OA={eval_report.overall_accuracy*100:.1f}%)",
    save_path=fig_path,
    figsize=(13, 11),
)
plt.show()
print(f"Confusion matrix plot saved to: {fig_path}")

Confusion matrix plot saved to: C:\Users\abhis\Desktop\Tetraspectra\outputs\figures\salinas_confusion_matrix.png


C:\Users\abhis\AppData\Local\Temp\ipykernel_4232\941808610.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Normalized Confusion Matrix (Recall per Class)
Normalizing each row by its ground-truth support reveals the recall rate and exact error leakage per class independent of sample size.

In [7]:
fig_norm_path = PROJECT_ROOT / "outputs" / "figures" / "salinas_confusion_matrix_normalized.png"

fig_norm = plot_confusion_matrix(
    cm=eval_report.confusion_matrix,
    class_names=[cm.class_name for cm in eval_report.per_class_metrics],
    normalize=True,
    title=f"Salinas 3D-CNN Normalized Confusion Matrix (Row Recall, AA={eval_report.average_accuracy*100:.1f}%)",
    save_path=fig_norm_path,
    figsize=(13, 11),
)
plt.show()

C:\Users\abhis\AppData\Local\Temp\ipykernel_4232\3549221287.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 8. Focused Error Analysis and Scientific Insights

### 1. The Primary Confusion: `Grapes_untrained` vs. `Vinyard_untrained`
Looking at the confusion matrix and per-class metrics:
- **Class 8 (`Grapes_untrained`)**: Precision: ~89.7%, Recall: ~97.9%
- **Class 15 (`Vinyard_untrained`)**: Precision: ~96.2%, Recall: ~82.5%

**Scientific Hypothesis**:
Why does the model confuse these two classes?
Both represent un-trellised grapevines in adjacent agricultural fields in the Salinas Valley. Both exhibit almost identical green canopy reflectance spectra in the NIR bands (around band 50 / 800nm) and identical dry soil background reflectance in the SWIR wavelengths. Because their spectral signatures and spatial textures are physically and agronomically nearly indistinguishable, the decision boundary between them exhibits the highest misclassification rate in the scene.

### 2. High-Performing Classes (>99% F1)
Classes like **Broccoli** (C1, C2), **Fallow** (C3, C4, C5), **Stubble** (C6), **Celery** (C7), and **Soil vineyard develop** (C9) achieve near 100% precision and recall. Their distinct diagnostic features (smooth plowed soil vs. dense high-moisture celery canopies) make them easy to separate in spatial-spectral 3D feature space.

### 3. Class Imbalance Impact
Notice that `Grapes_untrained` has 1,690 test samples while `Lettuce_romaine_6wk` has only 138 test samples (a 12:1 imbalance ratio). Because overall accuracy (OA = 97.02%) is sample-weighted, high performance on the large classes keeps OA elevated. However, **Average Accuracy (AA = 98.38%)** and **Macro F1 (98.49%)** verify that smaller classes are also classified accurately!

## 9. Persisting Evaluation Report to JSON
We serialize the complete evaluation report to `outputs/reports/day11_evaluation_report.json` for reproducible reporting.

In [8]:
report_json_path = PROJECT_ROOT / "outputs" / "reports" / "day11_evaluation_report.json"
saved_report_path = eval_report.save_json(report_json_path)
print(f"Persisted evaluation report to: {saved_report_path}")
print(f"File size: {saved_report_path.stat().st_size} bytes")

Persisted evaluation report to: C:\Users\abhis\Desktop\Tetraspectra\outputs\reports\day11_evaluation_report.json
File size: 7659 bytes


## 10. Summary & Interview Takeaways

| Metric / Concept | Measured Value | Practical Explanation |
|---|---|---|
| **Overall Accuracy (OA)** | **97.02%** | Proportion of total test pixels classified correctly ($7,878 / 8,120$). |
| **Average Accuracy (AA)** | **98.38%** | Unweighted mean of recall across all 16 classes; immune to class imbalance. |
| **Macro F1-Score** | **98.49%** | Harmonic mean of precision and recall averaged equally across classes. |
| **Cohen's Kappa ($\kappa$)** | **0.9668** | Statistical inter-rater agreement (>0.80 denotes "almost perfect agreement"). |
| **Weakest Class** | `Vinyard_untrained` (82.5% recall) | Misclassified into `Grapes_untrained` due to biophysical spectral similarity. |

---

### The Spatial Autocorrelation Caveat (Interview Essential)
In remote sensing, neighboring pixels share overlapping receptive fields ($7 \times 7$ window).
While random pixel-based stratified splitting is the established benchmark protocol for standard comparison on Salinas, Indian Pines, and Pavia University, overlapping spatial patches create subtle spatial correlation between training and test pixels.
For true operational deployment across independent flightlines, spatially disjoint block-based cross-validation is required.